# Road-Level AQI Estimation for Mumbai
## Stage 3: Land Use Regression (LUR) Feature Engineering

**Input:** `cleaned_aqi_mumbai_imputed.csv`  
**Output:** `lur_features.csv`  

This notebook enriches the cleaned AQI dataset with spatial, temporal, and environmental features required for a Land Use Regression model. All spatial features are computed per station (not per row), then joined back to the full timeseries.

### Feature categories built here
| # | Category | Source | Notes |
|---|----------|--------|-------|
| 1 | Road network density by type | OpenStreetMap (OSMnx) | Buffers: 100m, 300m, 500m, 1000m |
| 2 | Distance to nearest major road | OSM | Motorway / trunk / primary |
| 3 | Distance to point sources | OSM | Petrol stations, bus terminals, railway stations, airport |
| 4 | Land use type proportions | OSM | Industrial, commercial, residential, green space per buffer |
| 5 | Vegetation & green area | OSM + geometry | NDVI proxy, park area in 100m / 300m / 500m buffers |
| 6 | Distance to open water | OSM | Arabian Sea coastline, rivers, lakes |
| 7 | Population & building density | OSM building footprints + WorldPop proxy | 500m and 1000m buffers |
| 8 | Elevation | Open-Elevation API | Metres above sea level |
| 9 | Temporal features | Derived from datetime | Cyclic encoding, rush hour, season |

---

## 0. Imports & Setup

Install any missing packages before running:
```
pip install osmnx geopandas shapely requests tqdm
```

In [10]:
import pandas as pd
import numpy as np
import geopandas as gpd
import osmnx as ox
import requests
import warnings
import time
from shapely.geometry import Point, box
from shapely.ops import unary_union
from tqdm import tqdm

warnings.filterwarnings('ignore')

# ── OSMnx settings ────────────────────────────────────────────────────────────
# cache_folder stores downloaded OSM data locally so we only download once
ox.settings.use_cache = True
ox.settings.log_console = False

# ── Projection ────────────────────────────────────────────────────────────────
# EPSG:32643 = WGS84 / UTM Zone 43N — the correct projected CRS for Mumbai.
# All buffer and distance operations MUST be done in a projected (metric) CRS,
# not in geographic (lat/lon) coordinates, because degrees are not metres.
PROJECTED_CRS = 'EPSG:32643'
GEOGRAPHIC_CRS = 'EPSG:4326'

# ── Buffer sizes in metres ─────────────────────────────────────────────────────
BUFFERS = [100, 300, 500, 1000]

print('Libraries loaded.')
print(f'OSMnx version: {ox.__version__}')
print(f'GeoPandas version: {gpd.__version__}')

Libraries loaded.
OSMnx version: 2.1.0
GeoPandas version: 1.1.3


## 1. Load Data & Extract Unique Stations

Spatial features (road density, distances, land use) are **station-level constants** — they do not change hour-to-hour for a given station. We compute them once per station, then merge back onto the full timeseries at the end. This is far more efficient than computing them row-by-row.

In [11]:
# Load the full cleaned dataset
df = pd.read_csv('cleaned_aqi_mumbai_imputed.csv', parse_dates=['datetime'])

print(f'Full dataset shape : {df.shape}')
print(f'Date range         : {df["datetime"].min()} -> {df["datetime"].max()}')

# ── Extract unique stations ───────────────────────────────────────────────────
# Each station has a fixed lat/lon — this is what we compute spatial features for
stations = (
    df[['station_id', 'station_name', 'lat', 'lon']]
    .drop_duplicates(subset='station_id')
    .reset_index(drop=True)
)

print(f'\nUnique stations    : {len(stations)}')
stations

Full dataset shape : (100464, 21)
Date range         : 2025-07-01 00:00:00 -> 2025-12-31 23:00:00

Unique stations    : 27


,station_id,station_name,lat,lon
0,6927.0,"Colaba, Mumbai - MPCB-3379614",18.910000,72.820000
1,6945.0,"Kurla, Mumbai - MPCB-3379603",19.086300,72.888800
2,6948.0,"Chhatrapati Shivaji Intl. Airport (T2), Mumbai...",19.100780,72.874620
3,6956.0,"Powai, Mumbai - MPCB-3379610",19.137500,72.915056
4,6959.0,"Siddharth Nagar-Worli, Mumbai - IITM-3379674",19.000083,72.813993
5,6965.0,"Borivali East, Mumbai - MPCB-3397958",19.227474,72.864394
6,6967.0,"Sion, Mumbai - MPCB-3385877",19.047000,72.874600
7,6987.0,"Vile Parle West, Mumbai - MPCB-3379605",19.108610,72.836220
8,11606.0,"Borivali East, Mumbai - IITM-3379672",19.232410,72.868950
9,11611.0,"Malad West, Mumbai - IITM-3379673",19.197090,72.822040


In [12]:
# ── Create GeoDataFrame of stations ──────────────────────────────────────────
# Start in geographic CRS (lat/lon) for OSM queries, then project to UTM for
# all metric operations (buffers, distances)
stations_gdf = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations['lon'], stations['lat']),
    crs=GEOGRAPHIC_CRS
)

# Project to UTM Zone 43N for accurate metric calculations
stations_proj = stations_gdf.to_crs(PROJECTED_CRS)

print(f'Geographic CRS : {stations_gdf.crs}')
print(f'Projected CRS  : {stations_proj.crs}')
print(f'X range (metres from origin): {stations_proj.geometry.x.min():.0f} – {stations_proj.geometry.x.max():.0f}')

# ── Compute bounding box for OSM downloads ────────────────────────────────────
# Add 2km padding so features for edge stations are not clipped
pad = 0.02  # ~2km in degrees
BBOX = {
    'north': stations['lat'].max() + pad,
    'south': stations['lat'].min() - pad,
    'east':  stations['lon'].max() + pad,
    'west':  stations['lon'].min() - pad
}
print(f'\nBounding box for OSM downloads:')
for k, v in BBOX.items():
    print(f'  {k}: {v:.5f}')

Geographic CRS : EPSG:4326
Projected CRS  : EPSG:32643
X range (metres from origin): 269745 – 283576

Bounding box for OSM downloads:
  north: 19.31648
  south: 18.89000
  east: 72.96190
  west: 72.79281


---
## 2. Road Network Features

Road-related features are the most important predictors in any LUR model for urban air quality — vehicle emissions dominate urban PM2.5 and NO2. We differentiate by road type because a motorway 200m away contributes far more pollution than a residential street 200m away.

**Road type classification (OSM highway tags):**
| Category | OSM tags | Typical traffic |
|----------|----------|-----------------|
| Major | motorway, trunk, motorway_link, trunk_link | Very high — arterials & expressways |
| Secondary | primary, secondary, primary_link, secondary_link | Medium–high |
| Local | tertiary, residential, unclassified, living_street | Low |

**Features computed:**
- Road length density (m / m²) per type per buffer — how much road of each type is within the buffer
- Road network density (total length / buffer area) — overall urbanisation level
- Intersection count per buffer — more intersections = more idling, acceleration, braking
- Distance to nearest major road — strong negative gradient within 150m

In [13]:
# ── Download road network for Mumbai bounding box ─────────────────────────────
print('Downloading road network from OSM (drive network)...')
print('This may take 2-5 minutes. Data is cached after first download.')

G = ox.graph_from_bbox(
    bbox=(BBOX['west'], BBOX['south'], BBOX['east'], BBOX['north']),
    network_type='drive'
)

print(f'Graph downloaded: {len(G.nodes):,} nodes, {len(G.edges):,} edges')

# Convert to GeoDataFrames
nodes, edges = ox.graph_to_gdfs(G)

# Project to UTM for all metric operations
edges_proj = edges.to_crs(PROJECTED_CRS)
nodes_proj = nodes.to_crs(PROJECTED_CRS)

print(f'Edges CRS: {edges_proj.crs}')
print(f'Edges columns: {list(edges_proj.columns)}')

This may take 2-5 minutes. Data is cached after first download.
Graph downloaded: 30,898 nodes, 70,300 edges
Edges CRS: EPSG:32643
Edges columns: ['osmid', 'highway', 'lanes', 'name', 'oneway', 'reversed', 'length', 'maxspeed', 'geometry', 'tunnel', 'junction', 'bridge', 'access', 'width', 'ref']


In [14]:
# ── Classify roads by type ─────────────────────────────────────────────────────
# OSM 'highway' column contains the road type tag
# Some edges have lists (multiple tags) — take the first value
def flatten_highway(val):
    """Return highway tag as a string regardless of whether it is a list or string."""
    if isinstance(val, list):
        return val[0]
    return str(val)

edges_proj['highway_str'] = edges_proj['highway'].apply(flatten_highway)

# Define three tiers of road importance
MAJOR_ROADS    = {'motorway', 'trunk', 'motorway_link', 'trunk_link'}
SECONDARY_ROADS = {'primary', 'secondary', 'primary_link', 'secondary_link'}
LOCAL_ROADS    = {'tertiary', 'residential', 'unclassified', 'living_street', 'tertiary_link'}

edges_major     = edges_proj[edges_proj['highway_str'].isin(MAJOR_ROADS)].copy()
edges_secondary = edges_proj[edges_proj['highway_str'].isin(SECONDARY_ROADS)].copy()
edges_local     = edges_proj[edges_proj['highway_str'].isin(LOCAL_ROADS)].copy()

print(f'Major roads    : {len(edges_major):,} edges')
print(f'Secondary roads: {len(edges_secondary):,} edges')
print(f'Local roads    : {len(edges_local):,} edges')
print(f'Other/uncl.    : {len(edges_proj) - len(edges_major) - len(edges_secondary) - len(edges_local):,} edges')

Major roads    : 1,205 edges
Secondary roads: 9,964 edges
Local roads    : 59,131 edges
Other/uncl.    : 0 edges


In [15]:
# ── Extract road intersections from graph nodes ───────────────────────────────
# In OSMnx, nodes with degree >= 3 are real intersections (not dead ends or
# intermediate waypoints). More intersections = more stop-and-go traffic.
node_degrees   = dict(G.degree())
intersection_ids = [n for n, d in node_degrees.items() if d >= 3]
intersections  = nodes_proj.loc[nodes_proj.index.isin(intersection_ids)]

print(f'Total graph nodes  : {len(nodes_proj):,}')
print(f'Intersection nodes : {len(intersections):,}')

Total graph nodes  : 30,898
Intersection nodes : 24,689


In [16]:
# ── Compute road features for each station at each buffer ─────────────────────
print('Computing road network features per station...')

road_features = []

for idx, station in tqdm(stations_proj.iterrows(), total=len(stations_proj), desc='Stations'):
    row = {'station_id': station['station_id']}

    for dist in BUFFERS:
        buf = station.geometry.buffer(dist)  # Circular buffer of radius 'dist' metres
        area = buf.area                       # Buffer area in m²

        # ── Total road length density (all types combined) ────────────────────
        roads_in = edges_proj[edges_proj.intersects(buf)]
        # Clip to buffer boundary so we only count road length INSIDE the buffer
        roads_clipped = roads_in.copy()
        roads_clipped['geometry'] = roads_in.intersection(buf)
        total_len = roads_clipped.length.sum()
        row[f'road_density_total_{dist}m']   = total_len / area if area > 0 else 0
        row[f'road_len_total_{dist}m']        = total_len

        # ── Road length density by type ───────────────────────────────────────
        for road_type, road_gdf in [('major', edges_major),
                                      ('secondary', edges_secondary),
                                      ('local', edges_local)]:
            r_in = road_gdf[road_gdf.intersects(buf)]
            r_clip = r_in.copy()
            r_clip['geometry'] = r_in.intersection(buf)
            r_len = r_clip.length.sum()
            row[f'road_len_{road_type}_{dist}m']     = r_len
            row[f'road_density_{road_type}_{dist}m'] = r_len / area if area > 0 else 0

        # ── Intersection density ──────────────────────────────────────────────
        inter_in = intersections[intersections.intersects(buf)]
        row[f'intersection_count_{dist}m']   = len(inter_in)
        row[f'intersection_density_{dist}m'] = len(inter_in) / area if area > 0 else 0

    # ── Distance to nearest major road ───────────────────────────────────────
    # This is the single most important LUR predictor — distance from station
    # to the nearest motorway or trunk road
    if len(edges_major) > 0:
        dist_to_major = edges_major.distance(station.geometry).min()
    else:
        dist_to_major = np.nan
    row['dist_to_major_road_m'] = dist_to_major

    # ── Distance to nearest secondary road ───────────────────────────────────
    if len(edges_secondary) > 0:
        dist_to_secondary = edges_secondary.distance(station.geometry).min()
    else:
        dist_to_secondary = np.nan
    row['dist_to_secondary_road_m'] = dist_to_secondary

    road_features.append(row)

road_df = pd.DataFrame(road_features)
print(f'\nRoad features shape: {road_df.shape}')
print(f'Columns: {list(road_df.columns)}')

Computing road network features per station...


Stations: 100%|██████████| 27/27 [00:05<00:00,  5.31it/s]


Road features shape: (27, 43)
Columns: ['station_id', 'road_density_total_100m', 'road_len_total_100m', 'road_len_major_100m', 'road_density_major_100m', 'road_len_secondary_100m', 'road_density_secondary_100m', 'road_len_local_100m', 'road_density_local_100m', 'intersection_count_100m', 'intersection_density_100m', 'road_density_total_300m', 'road_len_total_300m', 'road_len_major_300m', 'road_density_major_300m', 'road_len_secondary_300m', 'road_density_secondary_300m', 'road_len_local_300m', 'road_density_local_300m', 'intersection_count_300m', 'intersection_density_300m', 'road_density_total_500m', 'road_len_total_500m', 'road_len_major_500m', 'road_density_major_500m', 'road_len_secondary_500m', 'road_density_secondary_500m', 'road_len_local_500m', 'road_density_local_500m', 'intersection_count_500m', 'intersection_density_500m', 'road_density_total_1000m', 'road_len_total_1000m', 'road_len_major_1000m', 'road_density_major_1000m', 'road_len_secondary_1000m', 'road_density_seconda

---
## 3. Point Source Distances

Specific point sources emit disproportionately large amounts of pollution. The distance from each station to these sources is a key LUR predictor:

- **Petrol stations:** VOC emissions from fuelling + concentrated road traffic
- **Bus terminals/depots:** Diesel buses are the largest single source of urban NOx in Mumbai
- **Railway stations:** Concentrate road-based pickup/drop-off traffic
- **Airport (CSIA T2):** Aircraft emissions + ground vehicles + jet fuel handling

All downloaded from OSM using `ox.features_from_bbox()`.

In [17]:
bbox_tuple = (BBOX['west'], BBOX['south'], BBOX['east'], BBOX['north'])

def download_osm_features(tags, desc):
    """Download OSM features within the bounding box, return as projected GeoDataFrame."""
    try:
        gdf = ox.features_from_bbox(
            bbox=(bbox_tuple[0], bbox_tuple[1], bbox_tuple[2], bbox_tuple[3]),
            tags=tags
        )
        # Use centroid for polygon features (e.g. large terminal complexes)
        gdf = gdf.copy()
        gdf['geometry'] = gdf.geometry.centroid
        gdf = gdf.to_crs(PROJECTED_CRS)
        print(f'  {desc}: {len(gdf):,} features downloaded')
        return gdf
    except Exception as e:
        print(f'  {desc}: FAILED — {e}. Returning empty GeoDataFrame.')
        return gpd.GeoDataFrame(geometry=[], crs=PROJECTED_CRS)

print('Downloading point source features from OSM...')

# Petrol/fuel stations
petrol_gdf = download_osm_features({'amenity': 'fuel'}, 'Petrol stations')

# Bus terminals and depots
bus_gdf = download_osm_features(
    {'amenity': ['bus_station'], 'highway': 'bus_stop'},
    'Bus terminals/stops'
)

# Railway stations
rail_gdf = download_osm_features(
    {'railway': ['station', 'halt']},
    'Railway stations'
)

# Airport (CSIA / Chhatrapati Shivaji Maharaj International Airport)
airport_gdf = download_osm_features(
    {'aeroway': ['aerodrome', 'terminal']},
    'Airport features'
)

  Petrol stations: 162 features downloaded
  Bus terminals/stops: 1,778 features downloaded
  Railway stations: 174 features downloaded
  Airport features: 9 features downloaded


In [18]:
def nearest_distance(station_geom, target_gdf, fallback=np.nan):
    """Return the minimum distance from station_geom to any feature in target_gdf."""
    if len(target_gdf) == 0:
        return fallback
    return target_gdf.geometry.distance(station_geom).min()

print('Computing distances to point sources...')

point_source_features = []

for idx, station in tqdm(stations_proj.iterrows(), total=len(stations_proj), desc='Stations'):
    row = {'station_id': station['station_id']}

    # Distance to nearest petrol/fuel station (metres)
    row['dist_petrol_station_m']  = nearest_distance(station.geometry, petrol_gdf)

    # Distance to nearest bus terminal or major bus stop (metres)
    row['dist_bus_terminal_m']    = nearest_distance(station.geometry, bus_gdf)

    # Distance to nearest railway station (metres)
    row['dist_railway_station_m'] = nearest_distance(station.geometry, rail_gdf)

    # Distance to airport (metres)
    # Airport is a large polygon — centroid approximation is sufficient for LUR
    row['dist_airport_m']         = nearest_distance(station.geometry, airport_gdf)

    # Count of petrol stations within 500m (intensity measure)
    buf_500 = station.geometry.buffer(500)
    row['petrol_count_500m'] = len(petrol_gdf[petrol_gdf.intersects(buf_500)]) if len(petrol_gdf) > 0 else 0

    point_source_features.append(row)

ps_df = pd.DataFrame(point_source_features)
print(f'\nPoint source features shape: {ps_df.shape}')

Computing distances to point sources...


Stations: 100%|██████████| 27/27 [00:00<00:00, 279.89it/s]


Point source features shape: (27, 6)


---
## 4. Land Use Type Features

Land use type is the definitional feature of a Land Use Regression model. Different land uses have different emission profiles:

- **Industrial:** Factories, warehouses, yards — major source of PM10, SO2, and heavy metals
- **Commercial:** Offices, shops, markets — vehicle deliveries, air conditioning
- **Residential:** Housing — cooking fuel, small generators, waste burning
- **Green/parks:** Vegetation absorbs pollutants and promotes dispersion — negative predictor

We compute the **proportion of each buffer area** covered by each land use type. A station surrounded by 60% industrial land within 500m will have systematically higher PM10 than one surrounded by 60% parks.

In [19]:
print('Downloading land use polygons from OSM...')

# OSM landuse tags covering the key LUR categories
landuse_gdf = ox.features_from_bbox(
    bbox=(bbox_tuple[0], bbox_tuple[1], bbox_tuple[2], bbox_tuple[3]),
    tags={'landuse': True}   # Download ALL landuse polygons
)

# Keep only Polygon/MultiPolygon geometries (not points or lines)
landuse_gdf = landuse_gdf[
    landuse_gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
].copy()

# Project to UTM
landuse_gdf = landuse_gdf.to_crs(PROJECTED_CRS)

print(f'Land use features downloaded: {len(landuse_gdf):,}')
print(f'Land use types present: {landuse_gdf["landuse"].value_counts().head(20).to_dict()}')

Land use features downloaded: 6,653
Land use types present: {'residential': 3732, 'commercial': 598, 'industrial': 484, 'grass': 429, 'construction': 409, 'recreation_ground': 265, 'farmland': 139, 'railway': 80, 'retail': 61, 'cemetery': 50, 'farmyard': 49, 'religious': 44, 'military': 42, 'education': 41, 'salt_pond': 39, 'greenfield': 37, 'aquaculture': 29, 'forest': 24, 'landfill': 19, 'meadow': 16}


In [20]:
# ── Group OSM landuse tags into meaningful categories ─────────────────────────
# OSM has many fine-grained tags — we aggregate into 4 LUR-relevant categories

LANDUSE_MAP = {
    'industrial': [
        'industrial', 'port', 'quarry', 'landfill', 'depot'
    ],
    'commercial': [
        'commercial', 'retail', 'warehouse', 'garages', 'construction'
    ],
    'residential': [
        'residential', 'apartments'
    ],
    'green': [
        'park', 'grass', 'forest', 'garden', 'nature_reserve',
        'recreation_ground', 'allotments', 'cemetery', 'village_green'
    ]
}

# Create a category column on the landuse GDF
def categorise_landuse(tag):
    if not isinstance(tag, str):
        return 'other'
    for category, tags in LANDUSE_MAP.items():
        if tag in tags:
            return category
    return 'other'

landuse_gdf['lu_category'] = landuse_gdf['landuse'].apply(categorise_landuse)

# Separate GDFs per category for efficient spatial operations
lu_industrial  = landuse_gdf[landuse_gdf['lu_category'] == 'industrial']
lu_commercial  = landuse_gdf[landuse_gdf['lu_category'] == 'commercial']
lu_residential = landuse_gdf[landuse_gdf['lu_category'] == 'residential']
lu_green       = landuse_gdf[landuse_gdf['lu_category'] == 'green']

print(f'Industrial parcels : {len(lu_industrial):,}')
print(f'Commercial parcels : {len(lu_commercial):,}')
print(f'Residential parcels: {len(lu_residential):,}')
print(f'Green parcels      : {len(lu_green):,}')

Industrial parcels : 510
Commercial parcels : 1,076
Residential parcels: 3,732
Green parcels      : 768


In [21]:
def landuse_area_in_buffer(lu_gdf, buffer_geom):
    """Return total area of land use polygons overlapping a buffer geometry."""
    if len(lu_gdf) == 0:
        return 0.0
    candidates = lu_gdf[lu_gdf.intersects(buffer_geom)]
    if len(candidates) == 0:
        return 0.0
    # Clip to buffer and sum clipped areas
    clipped = candidates.intersection(buffer_geom)
    return clipped.area.sum()

print('Computing land use proportions per station per buffer...')

# Use buffers of 300m, 500m, 1000m for land use
LU_BUFFERS = [300, 500, 1000]

lu_features = []

for idx, station in tqdm(stations_proj.iterrows(), total=len(stations_proj), desc='Stations'):
    row = {'station_id': station['station_id']}

    for dist in LU_BUFFERS:
        buf  = station.geometry.buffer(dist)
        area = buf.area  # Total buffer area in m²

        # Area and proportion of each land use type within the buffer
        for lu_name, lu_gdf in [
            ('industrial',  lu_industrial),
            ('commercial',  lu_commercial),
            ('residential', lu_residential),
            ('green',       lu_green)
        ]:
            lu_area = landuse_area_in_buffer(lu_gdf, buf)
            row[f'lu_{lu_name}_area_{dist}m']       = lu_area
            row[f'lu_{lu_name}_proportion_{dist}m'] = lu_area / area if area > 0 else 0

        # Total mapped land use area (industrial + commercial + residential + green)
        total_lu = sum(row[f'lu_{k}_area_{dist}m']
                       for k in ['industrial', 'commercial', 'residential', 'green'])
        row[f'lu_coverage_{dist}m'] = total_lu / area if area > 0 else 0

    lu_features.append(row)

lu_df = pd.DataFrame(lu_features)
print(f'\nLand use features shape: {lu_df.shape}')

Computing land use proportions per station per buffer...


Stations: 100%|██████████| 27/27 [00:01<00:00, 15.10it/s]


Land use features shape: (27, 28)


---
## 5. Vegetation, Green Area & Distance to Water

Vegetation reduces ambient particulate concentrations through dry deposition (particles stick to leaf surfaces) and promotes atmospheric dispersion. In Mumbai, there is strong spatial variation:
- **North Mumbai** (Borivali, near SGNP) — heavily vegetated
- **Central Mumbai** (Dharavi, Kurla) — minimal vegetation
- **South Mumbai** (Colaba, Worli) — moderate with some coastal parks

We use two approaches:
1. **OSM park/green area** — exact polygon intersection with buffers
2. **Distance to coast/water** — the Arabian Sea sea breeze is a key dispersion mechanism for western Mumbai stations; proximity to water is associated with lower AQI due to cleaner air advection

**Note on NDVI:** True NDVI requires satellite imagery (Sentinel-2 or Landsat). The OSM green area proportion serves as a reasonable proxy. If you want actual NDVI, it can be queried from Google Earth Engine with one API call.

In [22]:
print('Downloading green/vegetation features from OSM...')

# Parks, forests, recreation areas
parks_gdf = ox.features_from_bbox(
    bbox=(bbox_tuple[0], bbox_tuple[1], bbox_tuple[2], bbox_tuple[3]),
    tags={'leisure': ['park', 'garden', 'nature_reserve', 'recreation_ground',
                      'golf_course', 'pitch', 'playground']}
)
parks_gdf = parks_gdf[
    parks_gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
].to_crs(PROJECTED_CRS)
print(f'Parks/green spaces: {len(parks_gdf):,}')

# Natural features (mangroves, forests — important in coastal Mumbai)
natural_gdf = ox.features_from_bbox(
    bbox=(bbox_tuple[0], bbox_tuple[1], bbox_tuple[2], bbox_tuple[3]),
    tags={'natural': ['wood', 'scrub', 'grassland', 'wetland', 'tree_row']}
)
natural_gdf = natural_gdf[
    natural_gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
].to_crs(PROJECTED_CRS)
print(f'Natural areas: {len(natural_gdf):,}')

# Open water (sea, rivers, lakes, reservoirs)
water_gdf = ox.features_from_bbox(
    bbox=(bbox_tuple[0], bbox_tuple[1], bbox_tuple[2], bbox_tuple[3]),
    tags={'natural': 'water', 'waterway': True}
)
water_poly = water_gdf[
    water_gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
].to_crs(PROJECTED_CRS)
water_line = water_gdf[
    water_gdf.geometry.geom_type.isin(['LineString', 'MultiLineString'])
].to_crs(PROJECTED_CRS)
print(f'Water polygons: {len(water_poly):,}')
print(f'Water lines   : {len(water_line):,}')

Parks/green spaces: 2,062
Natural areas: 935
Water polygons: 525
Water lines   : 769


In [23]:
# Combine parks and natural green into a single green GDF
all_green_gdf = pd.concat([parks_gdf, natural_gdf], ignore_index=True)
# Combine water polygons and lines
all_water = pd.concat([water_poly, water_line], ignore_index=True)

print('Computing vegetation and water features per station...')

GREEN_BUFFERS = [100, 300, 500]   # metres

veg_features = []

for idx, station in tqdm(stations_proj.iterrows(), total=len(stations_proj), desc='Stations'):
    row = {'station_id': station['station_id']}

    # ── Green area proportion per buffer ──────────────────────────────────────
    for dist in GREEN_BUFFERS:
        buf  = station.geometry.buffer(dist)
        area = buf.area

        green_area = landuse_area_in_buffer(all_green_gdf, buf)
        row[f'green_area_{dist}m']       = green_area
        row[f'green_proportion_{dist}m'] = green_area / area if area > 0 else 0

    # ── Distance to nearest water body ────────────────────────────────────────
    # Covers Arabian Sea coastline, Ulhas River, Powai Lake, Vihar Lake, etc.
    if len(all_water) > 0:
        dist_water = all_water.geometry.distance(station.geometry).min()
    else:
        dist_water = np.nan
    row['dist_water_m'] = dist_water

    # ── Water area within 500m (coastal stations may have large water fraction) 
    buf_500 = station.geometry.buffer(500)
    water_area_500 = landuse_area_in_buffer(water_poly, buf_500)
    row['water_area_500m']       = water_area_500
    row['water_proportion_500m'] = water_area_500 / buf_500.area

    veg_features.append(row)

veg_df = pd.DataFrame(veg_features)
print(f'\nVegetation/water features shape: {veg_df.shape}')

Computing vegetation and water features per station...


Stations: 100%|██████████| 27/27 [00:00<00:00, 56.15it/s]


Vegetation/water features shape: (27, 10)


---
## 6. Population Density & Building Footprint Density

Population density is a proxy for total emission pressure — more people means more vehicles, more cooking fuel consumption, and more waste burning. Building density is a proxy for urban canyon effect — dense high-rise areas suppress wind speed and trap pollutants between buildings.

**Data source:** OSM building footprints (free, reasonable proxy for building density). For population density, we use a simplified proxy based on residential land use area since WorldPop rasters require a separate download. Instructions for WorldPop are commented below if you want true population counts.

In [24]:
print('Downloading building footprints from OSM...')

buildings_gdf = ox.features_from_bbox(
    bbox=(bbox_tuple[0], bbox_tuple[1], bbox_tuple[2], bbox_tuple[3]),
    tags={'building': True}  # All mapped buildings
)

# Keep only polygon geometries (actual building footprints, not nodes)
buildings_gdf = buildings_gdf[
    buildings_gdf.geometry.geom_type.isin(['Polygon', 'MultiPolygon'])
].to_crs(PROJECTED_CRS).copy()

print(f'Building footprints downloaded: {len(buildings_gdf):,}')

Building footprints downloaded: 104,376


In [25]:
# ── Optional: WorldPop population density ────────────────────────────────────
# To use actual population density raster from WorldPop (100m resolution):
# 1. Download from: https://www.worldpop.org/geodata/country/?id=IND
#    File: ind_ppp_2020_1km_Aggregated.tif (or 100m version)
# 2. pip install rasterio
# 3. Use rasterio to sample the raster at each station lat/lon
#
# For now we use building footprint area as the density proxy.

print('Computing building footprint density per station...')

BUILD_BUFFERS = [500, 1000]  # metres

build_features = []

for idx, station in tqdm(stations_proj.iterrows(), total=len(stations_proj), desc='Stations'):
    row = {'station_id': station['station_id']}

    for dist in BUILD_BUFFERS:
        buf  = station.geometry.buffer(dist)
        area = buf.area

        # Building footprint area within buffer
        build_area = landuse_area_in_buffer(buildings_gdf, buf)

        # Building count within buffer
        build_count = len(buildings_gdf[buildings_gdf.intersects(buf)])

        # Building coverage ratio (0-1): fraction of buffer covered by buildings
        # High BCR = dense urban canyon = lower wind speeds = higher pollution
        row[f'building_area_{dist}m']     = build_area
        row[f'building_count_{dist}m']    = build_count
        row[f'building_coverage_{dist}m'] = build_area / area if area > 0 else 0

        # Population proxy: residential building area within buffer
        # (scale by average household density if needed; here used as a relative measure)
        res_area = landuse_area_in_buffer(lu_residential, buf)
        row[f'residential_area_{dist}m']  = res_area
        row[f'population_proxy_{dist}m']  = res_area / area if area > 0 else 0

    build_features.append(row)

build_df = pd.DataFrame(build_features)
print(f'\nBuilding/population features shape: {build_df.shape}')

Computing building footprint density per station...


Stations: 100%|██████████| 27/27 [00:04<00:00,  6.61it/s]


Building/population features shape: (27, 11)


---
## 7. Elevation

Elevation affects pollution in two ways:
1. **Boundary layer dynamics:** Higher-elevation stations are closer to the top of the planetary boundary layer, where dispersion is more effective
2. **Local topography:** Stations in valleys or low-lying areas can experience pollution pooling

Mumbai's topography is relatively flat (0–10m across most of the city) but the Western Ghats influence flow patterns from the east. We query the Open-Elevation API which uses SRTM 30m data.

---
## 8. Temporal Features

Temporal features encode the time-varying context that determines emission patterns and atmospheric dispersion conditions. These are computed on the **full row-level dataset** (not per station), since they change with every hourly reading.

**Why cyclic encoding?**  
Hour 23 and hour 0 are adjacent in real time, but a raw integer treats them as far apart (23 vs 0). Encoding as sin/cos places them at adjacent positions on a circle, allowing any model to understand temporal proximity correctly.

| Feature | Encoding | Period |
|---------|----------|--------|
| Hour | sin + cos | 24 h |
| Weekday | sin + cos | 7 days |
| Month | sin + cos | 12 months |

In [27]:
print('Engineering temporal features on full dataset...')

# ── Extract raw time components ────────────────────────────────────────────────
df['hour']         = df['datetime'].dt.hour
df['minute']       = df['datetime'].dt.minute
df['day_of_week']  = df['datetime'].dt.dayofweek   # 0=Mon, 6=Sun
df['month']        = df['datetime'].dt.month
df['day_of_year']  = df['datetime'].dt.dayofyear
df['is_weekend']   = (df['day_of_week'] >= 5).astype(int)

# ── Cyclic encoding using sin/cos transformation ──────────────────────────────
# Formula: sin(2π * value / period), cos(2π * value / period)
# This maps the cyclic variable onto the unit circle, preserving temporal proximity.

# Hour of day (period = 24)
df['hour_sin']     = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos']     = np.cos(2 * np.pi * df['hour'] / 24)

# Day of week (period = 7)
df['weekday_sin']  = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['weekday_cos']  = np.cos(2 * np.pi * df['day_of_week'] / 7)

# Month (period = 12)
df['month_sin']    = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos']    = np.cos(2 * np.pi * df['month'] / 12)

# ── Rush hour flag ────────────────────────────────────────────────────────────
# Mumbai rush hours: morning 7-10 AM, evening 5-8 PM (17-20)
# This binary flag captures the traffic-driven emission spike
df['is_rush_hour'] = df['hour'].isin([7, 8, 9, 17, 18, 19, 20]).astype(int)

# ── Season (categorical + numerical) ─────────────────────────────────────────
season_map = {
    7: 'Monsoon', 8: 'Monsoon', 9: 'Monsoon',
    10: 'Post-Monsoon',
    11: 'Winter', 12: 'Winter'
}
df['season'] = df['month'].map(season_map)

# Encode season as ordinal (Monsoon=0, Post-Monsoon=1, Winter=2)
# This preserves the natural order from low to high pollution
season_ordinal = {'Monsoon': 0, 'Post-Monsoon': 1, 'Winter': 2}
df['season_code'] = df['season'].map(season_ordinal)

# Continuous time of day (decimal hours)
df['time_of_day']  = df['hour'] + df['minute'] / 60

print('Temporal features added:')
temporal_cols = ['hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour',
                 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos',
                 'month_sin', 'month_cos', 'season', 'season_code', 'time_of_day']
print(df[['datetime'] + temporal_cols].head(5).to_string())

Engineering temporal features on full dataset...
Temporal features added:
             datetime  hour  day_of_week  month  is_weekend  is_rush_hour  hour_sin  hour_cos  weekday_sin  weekday_cos  month_sin  month_cos   season  season_code  time_of_day
0 2025-07-01 00:00:00     0            1      7           0             0  0.000000  1.000000     0.781831      0.62349       -0.5  -0.866025  Monsoon            0          0.0
1 2025-07-01 01:00:00     1            1      7           0             0  0.258819  0.965926     0.781831      0.62349       -0.5  -0.866025  Monsoon            0          1.0
2 2025-07-01 02:00:00     2            1      7           0             0  0.500000  0.866025     0.781831      0.62349       -0.5  -0.866025  Monsoon            0          2.0
3 2025-07-01 03:00:00     3            1      7           0             0  0.707107  0.707107     0.781831      0.62349       -0.5  -0.866025  Monsoon            0          3.0
4 2025-07-01 04:00:00     4            1 

---
## 9. Merge All Spatial Features Back to Full Dataset

All the spatial features computed above are per-station constants. We now merge them all onto the full timeseries dataframe using `station_id` as the join key. Every hourly reading for station X will receive the same spatial features — those spatial characteristics do not change hour-to-hour.

In [28]:
# ── Merge all spatial feature tables ─────────────────────────────────────────
# Ensure station_id is the same dtype in all tables before merging
df['station_id'] = df['station_id'].astype(str)
for spatial_df in [road_df, ps_df, lu_df, veg_df, build_df, elev_df]:
    spatial_df['station_id'] = spatial_df['station_id'].astype(str)

df_enriched = df.copy()

# Merge each spatial feature table
merge_tables = [
    (road_df,  'Road network features'),
    (ps_df,    'Point source distances'),
    (lu_df,    'Land use proportions'),
    (veg_df,   'Vegetation & water'),
    (build_df, 'Building & population density'),
    (elev_df,  'Elevation'),
]

for spatial_df, name in merge_tables:
    before = len(df_enriched)
    df_enriched = df_enriched.merge(spatial_df, on='station_id', how='left')
    after = len(df_enriched)
    assert before == after, f'Row count changed after merging {name}!'
    print(f'Merged {name:35s} -> shape now {df_enriched.shape}')

print(f'\nFinal enriched dataset shape: {df_enriched.shape}')

Merged Road network features               -> shape now (100464, 79)
Merged Point source distances              -> shape now (100464, 84)
Merged Land use proportions                -> shape now (100464, 111)
Merged Vegetation & water                  -> shape now (100464, 120)
Merged Building & population density       -> shape now (100464, 130)
Merged Elevation                           -> shape now (100464, 131)

Final enriched dataset shape: (100464, 131)


---
## 10. Feature Audit & Quality Check

In [29]:
# ── Missing value audit ────────────────────────────────────────────────────────
total = len(df_enriched)
missing = df_enriched.isnull().sum()
missing_pct = (missing / total * 100).round(2)

missing_report = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).query('`Missing Count` > 0').sort_values('Missing %', ascending=False)

if missing_report.empty:
    print('No missing values in any feature column.')
else:
    print(f'Columns with missing values ({len(missing_report)}):')
    print(missing_report.to_string())

Columns with missing values (14):
             Missing Count  Missing %
elevation_m         100464     100.00
ws                   66301      65.99
wd                   65677      65.37
si_so2               10046      10.00
so2                  10046      10.00
pm25                  5804       5.78
si_pm25               5804       5.78
si_co                 5658       5.63
co                    5658       5.63
pm10                  2509       2.50
si_pm10               2509       2.50
no2                    882       0.88
si_no2                 882       0.88
aqi                    464       0.46


In [30]:
# ── Feature group summary ─────────────────────────────────────────────────────
groups = {
    'Core AQI / pollutants': ['pm25', 'pm10', 'no2', 'co', 'so2', 'aqi', 'aqi_category'],
    'Meteorological':        ['temp', 'rh', 'ws', 'wd'],
    'Road network':          [c for c in df_enriched.columns if 'road' in c or 'intersection' in c],
    'Point sources':         [c for c in df_enriched.columns if 'dist_' in c or 'petrol_count' in c],
    'Land use':              [c for c in df_enriched.columns if 'lu_' in c],
    'Vegetation / water':    [c for c in df_enriched.columns if 'green' in c or 'water' in c],
    'Buildings / population':[c for c in df_enriched.columns if 'building' in c or 'population' in c or 'residential_area' in c],
    'Elevation':             ['elevation_m'],
    'Temporal':              ['hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour',
                              'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos',
                              'month_sin', 'month_cos', 'season_code', 'time_of_day'],
}

print('Feature inventory:')
print('=' * 55)
total_features = 0
for group, cols in groups.items():
    present = [c for c in cols if c in df_enriched.columns]
    total_features += len(present)
    print(f'  {group:<30}: {len(present):>3} features')
print('=' * 55)
print(f'  {"Total features":<30}: {total_features:>3}')

Feature inventory:
  Core AQI / pollutants         :   7 features
  Meteorological                :   4 features
  Road network                  :  42 features
  Point sources                 :   8 features
  Land use                      :  27 features
  Vegetation / water            :  15 features
  Buildings / population        :  13 features
  Elevation                     :   1 features
  Temporal                      :  13 features
  Total features                : 130


In [31]:
# ── Sanity check: spatial features are constant per station ───────────────────
# Road density for a given station should be identical in every row
# If there is variation, the merge went wrong
check_col = [c for c in df_enriched.columns if 'road_density_total' in c]
if check_col:
    col = check_col[0]
    variance_per_station = (
        df_enriched.groupby('station_id')[col].std()
    )
    n_variable = (variance_per_station > 0).sum()
    if n_variable == 0:
        print(f'Sanity check PASSED: {col} is constant within each station.')
    else:
        print(f'WARNING: {n_variable} stations have non-constant {col} -- check merge!')

Sanity check PASSED: road_density_total_100m is constant within each station.


---
## 11. Export Final Feature-Engineered Dataset

In [32]:
# ── Define final column order ──────────────────────────────────────────────────
# Organise by logical group for readability

id_cols     = ['datetime', 'station_id', 'station_name', 'lat', 'lon']
target_cols = ['pm25', 'pm10', 'no2', 'co', 'so2', 'aqi', 'aqi_category']
met_cols    = ['temp', 'rh', 'ws', 'wd']

temp_cols   = ['hour', 'day_of_week', 'month', 'is_weekend', 'is_rush_hour',
               'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos',
               'month_sin', 'month_cos', 'season', 'season_code', 'time_of_day']

spatial_cols = (
    [c for c in df_enriched.columns if 'road'         in c] +
    [c for c in df_enriched.columns if 'intersection' in c] +
    [c for c in df_enriched.columns if 'dist_'        in c] +
    [c for c in df_enriched.columns if 'petrol_count' in c] +
    [c for c in df_enriched.columns if 'lu_'          in c] +
    [c for c in df_enriched.columns if 'green'        in c] +
    [c for c in df_enriched.columns if 'water'        in c] +
    [c for c in df_enriched.columns if 'building'     in c] +
    [c for c in df_enriched.columns if 'population'   in c] +
    [c for c in df_enriched.columns if 'residential_area' in c] +
    ['elevation_m']
)

# Deduplicate while preserving order
seen = set()
spatial_cols_unique = []
for c in spatial_cols:
    if c not in seen and c in df_enriched.columns:
        spatial_cols_unique.append(c)
        seen.add(c)

# si_ columns (sub-indices) are intermediate — exclude from the model dataset
final_cols = id_cols + target_cols + met_cols + temp_cols + spatial_cols_unique
final_cols = [c for c in final_cols if c in df_enriched.columns]

df_final = df_enriched[final_cols].copy()

print(f'Final dataset shape : {df_final.shape}')
print(f'Total columns       : {len(df_final.columns)}')
print(f'\nColumn list:')
for c in df_final.columns:
    print(f'  {c}')

Final dataset shape : (100464, 124)
Total columns       : 124

Column list:
  datetime
  station_id
  station_name
  lat
  lon
  pm25
  pm10
  no2
  co
  so2
  aqi
  aqi_category
  temp
  rh
  ws
  wd
  hour
  day_of_week
  month
  is_weekend
  is_rush_hour
  hour_sin
  hour_cos
  weekday_sin
  weekday_cos
  month_sin
  month_cos
  season
  season_code
  time_of_day
  road_density_total_100m
  road_len_total_100m
  road_len_major_100m
  road_density_major_100m
  road_len_secondary_100m
  road_density_secondary_100m
  road_len_local_100m
  road_density_local_100m
  road_density_total_300m
  road_len_total_300m
  road_len_major_300m
  road_density_major_300m
  road_len_secondary_300m
  road_density_secondary_300m
  road_len_local_300m
  road_density_local_300m
  road_density_total_500m
  road_len_total_500m
  road_len_major_500m
  road_density_major_500m
  road_len_secondary_500m
  road_density_secondary_500m
  road_len_local_500m
  road_density_local_500m
  road_density_total_1000m
  ro

In [33]:
# ── Export ────────────────────────────────────────────────────────────────────
OUTPUT_PATH = 'lur_features.csv'
df_final.to_csv(OUTPUT_PATH, index=False)

print(f'Saved: {OUTPUT_PATH}')
print(f'Shape: {df_final.shape}')
print()

# Final summary
print('=' * 55)
print('  FEATURE ENGINEERING COMPLETE')
print('=' * 55)
print(f'  Rows              : {len(df_final):,}')
print(f'  Total features    : {len(df_final.columns)}')
print(f'  Stations          : {df_final["station_id"].nunique()}')
print(f'  Date range        : {df_final["datetime"].min().date()} -> {df_final["datetime"].max().date()}')

remaining_na = df_final.isnull().sum().sum()
print(f'  Remaining NaNs    : {remaining_na}')
print('=' * 55)
print()
print('Next step: Load lur_features.csv in the LUR_Model_Training notebook.')

df_final.head(3)

Saved: lur_features.csv
Shape: (100464, 124)

  FEATURE ENGINEERING COMPLETE
  Rows              : 100,464
  Total features    : 124
  Stations          : 27
  Date range        : 2025-07-01 -> 2025-12-31
  Remaining NaNs    : 257805

Next step: Load lur_features.csv in the LUR_Model_Training notebook.


,datetime,station_id,station_name,lat,lon,pm25,pm10,no2,co,so2,...,building_count_500m,building_coverage_500m,building_area_1000m,building_count_1000m,building_coverage_1000m,population_proxy_500m,population_proxy_1000m,residential_area_500m,residential_area_1000m,elevation_m
0,2025-07-01 00:00:00,6927.0,"Colaba, Mumbai - MPCB-3379614",18.91,72.82,17.400000,54.6000,24.680000,0.600000,8.38,...,145,0.094292,261296.553164,575,0.083307,0.126594,0.090407,99267.440432,283566.569147,NaN
1,2025-07-01 01:00:00,6927.0,"Colaba, Mumbai - MPCB-3379614",18.91,72.82,18.218333,54.6000,24.613333,0.597500,8.38,...,145,0.094292,261296.553164,575,0.083307,0.126594,0.090407,99267.440432,283566.569147,NaN
2,2025-07-01 02:00:00,6927.0,"Colaba, Mumbai - MPCB-3379614",18.91,72.82,19.465556,36.6025,24.615000,0.598333,8.38,...,145,0.094292,261296.553164,575,0.083307,0.126594,0.090407,99267.440432,283566.569147,NaN
